In [ ]:
!pip install transformers torch scikit-learn -qq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Imports
import os
import random
import time
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, BertForSequenceClassification
from transformers.optimization import get_linear_schedule_with_warmup

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, precision_score
)

# Reproducibility
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Grabbed from other colab notebook with the baseline models
train = pd.read_csv('/content/drive/MyDrive/train.csv')
val   = pd.read_csv('/content/drive/MyDrive/val.csv')
test  = pd.read_csv('/content/drive/MyDrive/test.csv')

# df used only for visualizations later
df = pd.concat([train, val, test]).reset_index(drop=True)

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Total: {len(df):,}")
print(df['label'].value_counts())

Train: 613,788 | Val: 131,526 | Test: 131,526
Total: 876,840
label
1.0    471180
0.0    405660
Name: count, dtype: int64


In [ ]:
for split in [train, val, test]:
    split['label'] = split['label'].astype(int)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(texts):
    return tokenizer(list(texts), padding='max_length', truncation=True,
                     max_length=128, return_tensors='pt')

print("Tokenizing train...")
train_enc = tokenize_function(train['text'])
print("Tokenizing val...")
val_enc   = tokenize_function(val['text'])
print("Tokenizing test...")
test_enc  = tokenize_function(test['text'])

train_inputs, train_masks = train_enc['input_ids'], train_enc['attention_mask']
val_inputs,   val_masks   = val_enc['input_ids'],   val_enc['attention_mask']
test_inputs,  test_masks  = test_enc['input_ids'],  test_enc['attention_mask']

train_labels = torch.tensor(train['label'].values)
val_labels   = torch.tensor(val['label'].values)
test_labels  = torch.tensor(test['label'].values)

print(f"Train: {train_inputs.shape} | Val: {val_inputs.shape} | Test: {test_inputs.shape}")
print(f"Label distribution: {np.bincount(train_labels.numpy())}")

Tokenizing train...
Tokenizing val...
Tokenizing test...
Train: torch.Size([613788, 128]) | Val: torch.Size([131526, 128]) | Test: torch.Size([131526, 128])
Label distribution: [283962 329826]


In [ ]:
class RedditDepressionDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx]
        }

batch_size = 32

train_dataset = RedditDepressionDataset(train_inputs, train_masks, train_labels)
val_dataset   = RedditDepressionDataset(val_inputs,   val_masks,   val_labels)
test_dataset  = RedditDepressionDataset(test_inputs,  test_masks,  test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

loss_fn = torch.nn.CrossEntropyLoss()

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches:   {len(val_dataloader)}")
print(f"Test batches:  {len(test_dataloader)}")

Train batches: 19181
Val batches:   4111
Test batches:  4111


In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False,
)
model.to(device)
print(f"Model loaded on {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


In [ ]:
epochs = 2

optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-8)

total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def format_time(elapsed):
    return str(datetime.timedelta(seconds=int(round(elapsed))))

print(f"Total training steps: {total_steps}")

Total training steps: 38362


In [ ]:
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

best_val_loss = float('inf')
patience = 2
epochs_no_improve = 0

print('Starting training...')

for epoch_i in range(epochs):
    print(f'\n======== Epoch {epoch_i + 1} / {epochs} ========')
    print('Training...')

    t0 = time.time()
    total_train_loss = 0
    model.train()

    for step, batch in enumerate(train_dataloader):
        if step % 500 == 0 and step != 0:
            elapsed = format_time(time.time() - t0)
            print(f'  Batch {step:>5,} of {len(train_dataloader):>5,}. Elapsed: {elapsed}.')

        b_input_ids  = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels     = batch['labels'].to(device)

        model.zero_grad()
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        loss = loss_fn(outputs.logits, b_labels)
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f'  Average training loss: {avg_train_loss:.2f}')
    print(f'  Training epoch took: {format_time(time.time() - t0)}')

    print('\nValidating...')
    t0 = time.time()
    model.eval()
    total_eval_accuracy = 0
    total_eval_loss = 0

    for batch in val_dataloader:
        b_input_ids  = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels     = batch['labels'].to(device)

        with torch.no_grad():
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

        loss = loss_fn(outputs.logits, b_labels)
        total_eval_loss += loss.item()
        logits = outputs.logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(val_dataloader)
    avg_val_loss = total_eval_loss / len(val_dataloader)
    print(f'  Accuracy: {avg_val_accuracy:.4f}')
    print(f'  Validation Loss: {avg_val_loss:.4f}')
    print(f'  Validation took: {format_time(time.time() - t0)}')

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(avg_val_accuracy)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print('  Saved best model.')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f'  No improvement for {epochs_no_improve} epoch(s).')
        if epochs_no_improve >= patience:
            print('Early stopping triggered!')
            break

print('\nTraining complete!')

Starting training...

======== Epoch 1 / 2 ========
Training...
  Batch   500 of 19,181. Elapsed: 0:01:15.
  Batch 1,000 of 19,181. Elapsed: 0:02:29.
  Batch 1,500 of 19,181. Elapsed: 0:03:42.
  Batch 2,000 of 19,181. Elapsed: 0:04:56.
  Batch 2,500 of 19,181. Elapsed: 0:06:10.
  Batch 3,000 of 19,181. Elapsed: 0:07:23.
  Batch 3,500 of 19,181. Elapsed: 0:08:37.
  Batch 4,000 of 19,181. Elapsed: 0:09:51.
  Batch 4,500 of 19,181. Elapsed: 0:11:05.
  Batch 5,000 of 19,181. Elapsed: 0:12:18.
  Batch 5,500 of 19,181. Elapsed: 0:13:32.
  Batch 6,000 of 19,181. Elapsed: 0:14:46.
  Batch 6,500 of 19,181. Elapsed: 0:15:59.
  Batch 7,000 of 19,181. Elapsed: 0:17:13.
  Batch 7,500 of 19,181. Elapsed: 0:18:27.
  Batch 8,000 of 19,181. Elapsed: 0:19:40.
  Batch 8,500 of 19,181. Elapsed: 0:20:54.
  Batch 9,000 of 19,181. Elapsed: 0:22:08.
  Batch 9,500 of 19,181. Elapsed: 0:23:21.
  Batch 10,000 of 19,181. Elapsed: 0:24:35.
  Batch 10,500 of 19,181. Elapsed: 0:25:49.
  Batch 11,000 of 19,181. Elaps

In [ ]:
model.load_state_dict(torch.load('best_model.pt'))
print('Loaded best model weights.')

model.save_pretrained('/content/drive/MyDrive/bert_depression_model')
tokenizer.save_pretrained('/content/drive/MyDrive/bert_depression_model')
print('Saved to Drive.')

Loaded best model weights.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to Drive.


In [ ]:
print('Evaluating on Test Set...')
model.eval()

all_preds = []
all_labels = []

for batch in test_dataloader:
    b_input_ids  = batch['input_ids'].to(device)
    b_input_mask = batch['attention_mask'].to(device)
    b_labels     = batch['labels'].to(device)

    with torch.no_grad():
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    logits = outputs.logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()
    all_preds.extend(np.argmax(logits, axis=1).flatten())
    all_labels.extend(label_ids.flatten())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print(f'  Accuracy:  {np.sum(all_preds == all_labels) / len(all_labels):.4f}')
print(f'  Precision: {precision_score(all_labels, all_preds):.4f}')
print(f'  Recall:    {recall_score(all_labels, all_preds):.4f}')
print(f'  F1:        {f1_score(all_labels, all_preds):.4f}')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=['Non-depressed', 'Depressed']))

Evaluating on Test Set...
  Accuracy:  0.9435
  Precision: 0.9407
  Recall:    0.9550
  F1:        0.9478

Classification Report:
               precision    recall  f1-score   support

Non-depressed       0.95      0.93      0.94     60849
    Depressed       0.94      0.96      0.95     70677

     accuracy                           0.94    131526
    macro avg       0.94      0.94      0.94    131526
 weighted avg       0.94      0.94      0.94    131526

